## Data Cleaning

### Objective

The objective of this notebook is to clean the synthetic ISP customer dataset before performing exploratory data analysis and machine learning.

The cleaning process follows business-oriented rules instead of blindly applying generic preprocessing techniques.

The following tasks will be performed:

- Remove duplicate records
- Handle missing values
- Standardize categorical labels
- Correct invalid bandwidth values
- Validate data quality
- Save the cleaned dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/synthetic_isp_customers_v2.csv")

## Load Dataset

The raw synthetic dataset is loaded for cleaning.

In [2]:
df.head()

,Customer_ID,Bandwidth_Mbps,Price_Per_Mbps,Monthly_Internet_Fee,Years_With_Company,Customer_Type,Company_Size,Strategic_Category,Strategic_Level,Has_VoIP,Monthly_VoIP_Fee,Has_Server,Server_Type,Monthly_Server_Fee,Support_Tickets_Last_Year,Downtime_Hours_Last_Year,Contract_Type
0,C00001,10,320000,3200000,4.2,Personal,Small,Normal,0,0,NaN,0,NaN,NaN,0,4.8,Monthly
1,C00002,10,385000,3850000,1.5,Personal,Small,Normal,0,0,NaN,0,NaN,NaN,3,4.8,Monthly
2,C00003,20,335000,6700000,0.9,Personal,Small,Normal,0,0,NaN,0,NaN,NaN,3,5.6,Monthly
3,C00004,50,315000,15750000,3.1,Personal,Small,Normal,0,0,NaN,0,NaN,NaN,2,1.3,Monthly
4,C00005,30,350000,10500000,2.2,Personal,Small,Normal,0,0,NaN,0,NaN,NaN,5,6.3,Monthly


## 1. Remove Duplicate Records

Duplicate customer records may bias future analyses and machine learning models.

Each customer should appear only once in the dataset.

In [3]:
duplicates = df.duplicated().sum()

print(f"Duplicate records before cleaning: {duplicates}")

Duplicate records before cleaning: 8


### Business Rule

This dataset represents a single snapshot of ISP customers rather than historical transactions.

Therefore, each customer should appear only once. Exact duplicate records are considered data quality issues and will be removed.

In [4]:
df = df.drop_duplicates()

In [5]:
duplicates = df.duplicated().sum()

print(f"Duplicate records after cleaning: {duplicates}")

Duplicate records after cleaning: 0


### Verification

Duplicate records were successfully removed from the dataset.

## 2. Handle Missing Values

Missing values should not be treated equally.

Some missing values are expected because certain customers do not subscribe to optional services, while others indicate incomplete information that should be imputed.

Each column will therefore be handled according to its business meaning.

In [6]:
missing = df.isnull().sum()

missing = missing[missing > 0]

missing

Years_With_Company            25
Monthly_VoIP_Fee            2029
Server_Type                 2380
Monthly_Server_Fee          2380
Downtime_Hours_Last_Year      30
dtype: int64

### Business Rule

Years with the company is an important feature for customer prioritization.

Missing values are assumed to result from incomplete records rather than unavailable information.

The missing values will therefore be replaced using the median value.

In [16]:
missing = df.isnull().sum()

missing = missing[missing > 0]

missing

Monthly_VoIP_Fee            2029
Server_Type                 2380
Monthly_Server_Fee          2380
Downtime_Hours_Last_Year      30
dtype: int64

### 2.1 Years_With_Company

#### Business Rule

Years with the company is an important feature for customer prioritization.

The missing values are assumed to result from incomplete records rather than unavailable information.

Since the number of missing values is very small, they will be replaced using the median, which is more robust to outliers than the mean.

In [17]:
median_years = df["Years_With_Company"].median()

median_years

np.float64(5.1)

In [18]:
df["Years_With_Company"] = df["Years_With_Company"].fillna(median_years)

In [19]:
df["Years_With_Company"].isnull().sum()

np.int64(0)

### Verification

All missing values in the **Years_With_Company** column have been successfully replaced with the median value.

### 2.2 Monthly_VoIP_Fee

#### Business Rule

Customers who do not subscribe to the VoIP service should not have a monthly VoIP fee.

Therefore, missing values in this column are considered valid business cases rather than missing information.

These values will be replaced with zero.

In [21]:
df[df["Monthly_VoIP_Fee"].isna()]["Has_VoIP"].value_counts(dropna=False)

Has_VoIP
0    2029
Name: count, dtype: int64

In [22]:
df["Monthly_VoIP_Fee"] = df["Monthly_VoIP_Fee"].fillna(0)

In [23]:
df["Monthly_VoIP_Fee"].isnull().sum()

np.int64(0)

### Verification

All customers without a VoIP subscription now have a monthly VoIP fee of zero.

Since these customers do not purchase the service, zero correctly represents the business meaning of the data.

### 2.3 Server Service Information

#### Business Rule

The server-related columns are evaluated together because they describe the same business service.

Customers without a server subscription should not have a monthly server fee.

Therefore:

- Missing values in **Monthly_Server_Fee** will be replaced with zero.
- Missing values in **Server_Type** will remain unchanged because the absence of a server type correctly represents customers who do not subscribe to this service.

In [24]:
df[df["Server_Type"].isna()]["Has_Server"].value_counts(dropna=False)

Has_Server
0    2380
Name: count, dtype: int64

In [25]:
df[df["Monthly_Server_Fee"].isna()]["Has_Server"].value_counts(dropna=False)

Has_Server
0    2380
Name: count, dtype: int64

In [26]:
df["Monthly_Server_Fee"] = df["Monthly_Server_Fee"].fillna(0)

In [27]:
print("Missing values in Server_Type:")
print(df["Server_Type"].isnull().sum())

print("\nMissing values in Monthly_Server_Fee:")
print(df["Monthly_Server_Fee"].isnull().sum())

Missing values in Server_Type:
2380

Missing values in Monthly_Server_Fee:
0


### Verification

The monthly server fee has been successfully updated for customers without a server subscription.

The **Server_Type** column intentionally retains missing values because these customers do not use any server service. Keeping these values as missing preserves the original business meaning of the data.

### 2.4 Downtime_Hours_Last_Year

#### Business Rule

Downtime information is an important service quality indicator.

Missing values do not necessarily indicate zero downtime.

To avoid introducing incorrect assumptions, missing values will be replaced using the median downtime.

In [30]:
median_downtime = df["Downtime_Hours_Last_Year"].median()

median_downtime

np.float64(4.7)

In [31]:
df["Downtime_Hours_Last_Year"] = df["Downtime_Hours_Last_Year"].fillna(median_downtime)

In [32]:
df["Downtime_Hours_Last_Year"].isnull().sum()

np.int64(0)

### Missing Value Validation

The dataset is validated after handling missing values to ensure that only intentional missing values remain.

In [33]:
df.isnull().sum()

Customer_ID                     0
Bandwidth_Mbps                  0
Price_Per_Mbps                  0
Monthly_Internet_Fee            0
Years_With_Company              0
Customer_Type                   0
Company_Size                    0
Strategic_Category              0
Strategic_Level                 0
Has_VoIP                        0
Monthly_VoIP_Fee                0
Has_Server                      0
Server_Type                  2380
Monthly_Server_Fee              0
Support_Tickets_Last_Year       0
Downtime_Hours_Last_Year        0
Contract_Type                   0
dtype: int64

### Validation Summary

The missing value handling process has been completed.

All missing values have been addressed according to their business meaning.

The only remaining missing values belong to the **Server_Type** column. These values are intentionally preserved because customers without a server subscription do not have a valid server type.

The dataset is now ready for the next stage of data cleaning.

## 3. Standardize Categorical Values

### Objective

Categorical variables should contain consistent labels.

In real-world datasets, the same category may appear in different formats due to manual data entry, such as differences in capitalization or spelling.

Before applying any corrections, all unique values will be inspected.

In [34]:
categorical_columns = [
    "Customer_Type",
    "Company_Size",
    "Strategic_Category",
    "Contract_Type"
]

categorical_columns

['Customer_Type', 'Company_Size', 'Strategic_Category', 'Contract_Type']

### Inspect Unique Values

The unique values of each categorical feature are displayed to identify inconsistent labels before standardization.

In [36]:
df["Customer_Type"].value_counts(dropna=False)

Customer_Type
Personal      1487
Business      1251
Government     261
Goverment        1
Name: count, dtype: int64

In [37]:
df["Company_Size"].value_counts(dropna=False)

Company_Size
Small     1875
Large      732
Medium     393
Name: count, dtype: int64

In [38]:
df["Strategic_Category"].value_counts(dropna=False)

Strategic_Category
Normal            1487
Small Business     779
Government         262
Enterprise         132
University         117
ISP Partner        112
Hospital           109
SmallBusiness        2
Name: count, dtype: int64

In [39]:
df["Contract_Type"].value_counts(dropna=False)

Contract_Type
Monthly    1866
Annual     1134
Name: count, dtype: int64

### 3.1 Customer_Type

#### Business Rule

Customer categories should follow a consistent naming convention.

The value **"Goverment"** is a spelling mistake and will be standardized to **"Government"**.

In [40]:
df["Customer_Type"] = df["Customer_Type"].replace({
    "Goverment": "Government"
})

In [41]:
df["Customer_Type"].value_counts()

Customer_Type
Personal      1487
Business      1251
Government     262
Name: count, dtype: int64

### 3.2 Strategic_Category

#### Business Rule

Strategic customer categories should use consistent labels.

The value **"SmallBusiness"** represents the same category as **"Small Business"** and will therefore be standardized.

In [42]:
df["Strategic_Category"] = df["Strategic_Category"].replace({
    "SmallBusiness": "Small Business"
})

In [43]:
df["Strategic_Category"].value_counts()

Strategic_Category
Normal            1487
Small Business     781
Government         262
Enterprise         132
University         117
ISP Partner        112
Hospital           109
Name: count, dtype: int64

### Validation Summary

All identified inconsistencies in categorical labels have been corrected.

The categorical variables now follow a consistent naming convention and are ready for further analysis and machine learning.

## 4. Invalid Value Correction

### Objective

Some numerical values may not follow the predefined business rules.

Instead of removing these records, invalid values will be corrected according to the ISP's available service plans while preserving as much information as possible.

### 4.1 Bandwidth_Mbps

#### Business Rule

The ISP offers a fixed set of bandwidth plans.

Any value outside these predefined plans is considered an invalid entry and will be mapped to the nearest valid bandwidth plan instead of being removed.

In [44]:
valid_bandwidth = [
    10, 20, 30, 40, 50,
    75, 100, 150, 200, 300
]

valid_bandwidth

[10, 20, 30, 40, 50, 75, 100, 150, 200, 300]

In [45]:
invalid_bandwidth = df[
    ~df["Bandwidth_Mbps"].isin(valid_bandwidth)
]

invalid_bandwidth

,Customer_ID,Bandwidth_Mbps,Price_Per_Mbps,Monthly_Internet_Fee,Years_With_Company,Customer_Type,Company_Size,Strategic_Category,Strategic_Level,Has_VoIP,Monthly_VoIP_Fee,Has_Server,Server_Type,Monthly_Server_Fee,Support_Tickets_Last_Year,Downtime_Hours_Last_Year,Contract_Type
1803,C01804,250,300000,6000000,6.3,Personal,Small,Normal,0,0,0.0,0,NaN,0.0,2,6.8,Monthly
2044,C02045,250,255000,25500000,8.8,Business,Medium,Small Business,1,1,1069952.0,0,NaN,0.0,6,10.8,Annual
2058,C02059,250,270000,13500000,3.8,Business,Small,Small Business,1,0,0.0,1,Dedicated,13728753.0,6,6.2,Monthly
2203,C02204,250,275000,27500000,0.2,Business,Medium,Small Business,1,1,469774.0,0,NaN,0.0,5,3.8,Monthly
2409,C02410,250,230000,34500000,2.5,Business,Small,Small Business,1,0,0.0,0,NaN,0.0,0,1.9,Monthly


In [46]:
print(f"Number of invalid bandwidth values: {len(invalid_bandwidth)}")

Number of invalid bandwidth values: 5


In [47]:
def nearest_bandwidth(value):

    return min(
        valid_bandwidth,
        key=lambda x: abs(x - value)
    )

In [48]:
df["Bandwidth_Mbps"] = df["Bandwidth_Mbps"].apply(
    nearest_bandwidth
)

In [49]:
invalid_bandwidth_after = df[
    ~df["Bandwidth_Mbps"].isin(valid_bandwidth)
]

print(f"Invalid bandwidth values after correction: {len(invalid_bandwidth_after)}")

Invalid bandwidth values after correction: 0


### Verification

All invalid bandwidth values have been mapped to the nearest valid ISP bandwidth plan.

This approach preserves customer records while ensuring that every bandwidth value complies with the company's predefined service offerings.

## 5. Data Type Validation

### Objective

Before proceeding to exploratory data analysis and machine learning, the data types of all features are validated.

Correct data types improve memory efficiency and ensure that each feature is interpreted correctly during analysis.

In [50]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Customer_ID                3000 non-null   str    
 1   Bandwidth_Mbps             3000 non-null   int64  
 2   Price_Per_Mbps             3000 non-null   int64  
 3   Monthly_Internet_Fee       3000 non-null   int64  
 4   Years_With_Company         3000 non-null   float64
 5   Customer_Type              3000 non-null   str    
 6   Company_Size               3000 non-null   str    
 7   Strategic_Category         3000 non-null   str    
 8   Strategic_Level            3000 non-null   int64  
 9   Has_VoIP                   3000 non-null   int64  
 10  Monthly_VoIP_Fee           3000 non-null   float64
 11  Has_Server                 3000 non-null   int64  
 12  Server_Type                620 non-null    str    
 13  Monthly_Server_Fee         3000 non-null   float64
 14  Sup

### Validation

The data types of all features were reviewed.

All columns use appropriate data types based on their business meaning.

The only column containing missing values is **Server_Type**, which intentionally remains incomplete because customers without a server subscription do not have a corresponding server type.

No additional data type conversions were required.

## 6. Save Clean Dataset

### Objective

After completing all data cleaning steps, the cleaned dataset is saved for future analysis.

Keeping the raw and cleaned datasets separate ensures reproducibility and preserves the original data.

In [54]:
df.to_csv(
    "../data/cleaned_isp_customers.csv",
    index=False
)

In [55]:
import os

os.path.exists("../data/cleaned_isp_customers.csv")

True

### Verification

The cleaned dataset has been successfully exported and is now ready for exploratory data analysis (EDA) and machine learning.

## Conclusion

The dataset has been successfully cleaned and prepared for further analysis.

The following preprocessing tasks were completed:

- Removed duplicate records
- Handled missing values using business-oriented rules
- Standardized inconsistent categorical labels
- Corrected invalid bandwidth values
- Validated data types

The cleaned dataset is now ready for exploratory data analysis, feature engineering, customer scoring, and clustering.